In [ ]:
from tqdm.notebook import trange
from tqdm import tqdm
import numpy as np
import scipy as sp
import time
import math
import random
import pandas as pd
from matplotlib import pyplot as plt
import h5py
from pathlib import Path
import logging
import sys
import subprocess

%matplotlib inline

In [ ]:
def my_fsr_cmd(target, cmd, scmd, data, payload_only=False):
    target.flush()
    target.send_cmd(cmd=cmd, scmd=ord(scmd), data=data)

    response = target.read_cmd(timeout=500)

    if payload_only:
        return response[3:3 + response[2]]

    return response

In [ ]:
def reset_target(scope):
    scope.io.nrst = 'low'
    time.sleep(0.25)
    scope.io.nrst = 'high_z'
    time.sleep(0.25)

In [ ]:
# Bokeh 시각화 초기화 (한 번만 실행하면 노트북 전체에서 인라인 출력 가능)
from bokeh.plotting import figure, show
from bokeh.io import output_notebook
from bokeh.models import HoverTool, ColumnDataSource, Range1d
from bokeh.palettes import Category10, Viridis256
from bokeh.layouts import column

output_notebook()

def plot_t(traces, num_plot=10, title=None, 
                       width=900, height=360, show_plot=True):
    """다중 파형을 겹쳐 그리는 Bokeh 시각화."""
    t = np.asarray(traces)
    x = np.arange(t.shape[1])
    
    p = figure(
        width=width, height=height,
        title=title or f'Traces',
        x_axis_label='Sample', y_axis_label='Amplitude',
        tools='pan,wheel_zoom,box_zoom,reset,save',
        active_scroll='wheel_zoom',
        background_fill_color='#fafafa',
        border_fill_color='white',
    )
    
    palette = Category10[10]
    for i in range(num_plot):
        p.line(x, t[i], line_width=1.0, line_color=palette[i % 10],
               line_alpha=0.75, legend_label=f'Index {i}')
    
    # 시각적 다듬기
    p.title.text_font_size = '13pt'
    p.title.text_color = '#2c3e50'
    p.title.align = 'center'
    p.grid.grid_line_alpha = 0.3
    p.xaxis.axis_label_text_font_style = 'normal'
    p.yaxis.axis_label_text_font_style = 'normal'
    p.outline_line_color = None
    
    # 범례 설정
    p.legend.location = 'top_right'
    p.legend.click_policy = 'hide'
    p.legend.label_text_font_size = '9pt'
    p.legend.background_fill_alpha = 0.7
    
    # Hover 툴팁
    p.add_tools(HoverTool(tooltips=[('Sample', '$x{0}'), 
                                     ('Amplitude', '$y{0.0000}')]))
    
    if show_plot:
        show(p)
    return p